# 8.6 · 投票与平均 / Voting & Averaging

> **课程定位 / Where this fits**
> 第 6 课，**Part 8 · 集成学习**。
> Lesson 6, **Part 8 · Ensemble Learning**.
>
> 前几课的集成都是**同质的**（一堆决策树）。从这一课起做**异质集成**——把**完全不同的模型**（逻辑回归 + SVM + 随机森林 + …）组合起来。最简单的组合方式就是 **voting（投票）/ averaging（平均）**。核心直觉：不同模型犯**不同的错**，组合起来错误会部分抵消——前提是它们**足够多样且各自够好**。
> Earlier ensembles were **homogeneous** (many trees). Now **heterogeneous ensembles** — combining **completely different models** (logistic + SVM + random forest + …). The simplest combination is **voting / averaging**. The core intuition: different models make **different mistakes**, so combining them partially cancels errors — provided they're **diverse enough and each decent**.
>
> 💼 **实战/面试视角**："硬投票 vs 软投票 / 集成为什么有效 / 模型要多样" 是集成进阶常考。
> 💼 **Practical/interview angle:** "hard vs soft voting / why ensembles work / model diversity" — common.

> 📐 **符号约定 / Notation**
> - 硬投票 hard voting —— 按预测类别投票（少数服从多数）/ vote on predicted classes
> - 软投票 soft voting —— 按预测概率平均 / average predicted probabilities

> 💡 **面试相关 / Interview-relevant**
> - "硬投票 vs 软投票的区别"（出镜率 ★★★★★）
> - "集成为什么有效（多样性 + 各自够好）"（★★★★★）
> - "voting 比单模型一定好吗"（★★★★，不一定）
> - "怎么给 voting 加权"（★★★）

---

## 学习目标 / Learning Objectives

1. 理解异质集成 + 多样性为何关键。
   Understand heterogeneous ensembles and why diversity matters.
2. 区分**硬投票**与**软投票**。
   Distinguish hard vs soft voting.
3. 看集成的多样性（模型预测相关性低）。
   See ensemble diversity (low correlation of model predictions).
4. 用加权投票并知道它不总比最好的单模型强。
   Use weighted voting and know it doesn't always beat the best single model.

## 目录 / TOC
1. [先建直觉：为什么集成有效 ⭐](#1)
2. [🍷 数据 + 多样的基模型](#2)
3. [硬投票 vs 软投票 ⭐](#3)
4. [多样性是关键 ⭐](#4)
5. [加权投票 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉：为什么集成有效 ⭐ / Why Ensembles Work

设想三个模型各自准确率 70%，但它们**犯的错不一样**（在不同样本上出错）。对某个样本多数投票时，只有当**至少两个**同时错才会错——如果错误**互相独立**，三个里两个同时错的概率远小于单个 30% 的错误率。于是集成准确率能**高于任何单个模型**。
Imagine three models each 70% accurate but making **different mistakes** (wrong on different samples). Majority voting errs only when **at least two** are wrong at once — if errors are **independent**, two-of-three being wrong is far less likely than a single 30% error. So the ensemble can be **more accurate than any single model**.

**两个必要条件**（面试核心）：
**Two necessary conditions** (the core point):
1. **各自够好**：每个模型至少比瞎猜强（否则一起投票只会更差）。
   **Each is decent:** every model is better than chance (else voting makes it worse).
2. **足够多样**：模型们犯**不同**的错（错误相关性低）。把 5 个几乎一样的模型放一起投票，毫无意义。
   **Diverse enough:** models make **different** errors (low error correlation). Voting 5 near-identical models is pointless.

多样性从哪来？用**不同算法**（线性 + 核 + 树）、不同特征子集、不同超参——异质集成天然多样。
Where does diversity come from? **Different algorithms** (linear + kernel + tree), feature subsets, hyperparameters — heterogeneous ensembles are naturally diverse.


<a id="2"></a>
## 2. 数据 + 多样的基模型 / Data & Diverse Base Models

用 **Wine**（13 特征，3 类）。准备三个**原理迥异**的基模型：逻辑回归（线性）、SVM（核）、随机森林（树集成）——它们的归纳偏置不同，会犯不同的错，正适合做投票集成。
Using **Wine** (13 features, 3 classes). We prepare three **fundamentally different** base models: logistic regression (linear), SVM (kernel), random forest (tree ensemble) — different inductive biases, different mistakes, ideal for a voting ensemble.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
sns.set_theme(style="whitegrid")

wine = load_wine()
X_tr, X_te, y_tr, y_te = train_test_split(wine.data, wine.target, test_size=0.3,
                                          stratify=wine.target, random_state=0)

# 三个原理迥异的基模型(线性/核/树); 距离类模型放进 Pipeline 缩放 / three diverse base models
base = {
    "逻辑回归 LogReg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "SVM(rbf)":        make_pipeline(StandardScaler(), SVC(probability=True, random_state=0)),
    "随机森林 RF":      RandomForestClassifier(n_estimators=200, random_state=0),
}
print("各基模型单独的 CV 准确率:")
for name, m in base.items():
    print(f"  {name:<16} {cross_val_score(m, wine.data, wine.target, cv=5).mean():.3f}")


<a id="3"></a>
## 3. 硬投票 vs 软投票 ⭐ / Hard vs Soft Voting

`VotingClassifier` 两种模式（面试必区分）：
`VotingClassifier` has two modes (must distinguish):
- **硬投票(hard)**：每个模型给出一个**类别**，少数服从多数。简单，但**丢掉了置信度信息**（"勉强 51%" 和 "笃定 99%" 一样算一票）。
  **Hard voting:** each model outputs a **class**, majority wins. Simple, but **discards confidence** ("barely 51%" counts the same as "confident 99%").
- **软投票(soft)**：把各模型的**预测概率平均**，取概率最大的类。它**用上了置信度**——一个很自信的模型能压过两个犹豫的模型。**通常软投票更好**，前提是各模型的概率**校准**得还行(7.9)。
  **Soft voting:** **average the predicted probabilities** and take the argmax. It **uses confidence** — one confident model can outweigh two hesitant ones. **Soft voting is usually better**, provided the models' probabilities are reasonably calibrated (7.9).


In [ ]:
from sklearn.ensemble import VotingClassifier

estimators = list(base.items())                       # [(名字, 模型), ...]
hard = VotingClassifier(estimators, voting="hard")    # 按类别投票
soft = VotingClassifier(estimators, voting="soft")    # 按概率平均(需 predict_proba)

print(f"{'方法':<22}{'CV 准确率':>10}")
for name, m in base.items():
    print(f"{name:<24}{cross_val_score(m, wine.data, wine.target, cv=5).mean():>10.3f}")
print("-"*34)
print(f"{'硬投票 hard voting':<24}{cross_val_score(hard, wine.data, wine.target, cv=5).mean():>10.3f}")
print(f"{'软投票 soft voting':<24}{cross_val_score(soft, wine.data, wine.target, cv=5).mean():>10.3f}")
print("\n软投票通常≥硬投票(用上了概率置信度); 集成往往≥最好的单模型(若基模型多样)")


<a id="4"></a>
## 4. 多样性是关键 ⭐ / Diversity Is the Key

验证"多样性"这个核心条件：算三个基模型在测试集上**预测的两两相关性**。相关性越低 = 它们犯的错越不一样 = 集成收益越大。再做个反面对照——把**三个几乎一样的模型**（同算法不同随机种子）投票，几乎没有提升。
We verify the core condition "diversity": compute the **pairwise correlation** of the three base models' test predictions. Lower correlation = more different mistakes = bigger ensemble gain. As a counter-example, voting **three near-identical models** (same algorithm, different seeds) gives almost no lift.


In [ ]:
# 各基模型在测试集上的预测, 算两两相关性 / pairwise correlation of predictions
preds = {}
for name, m in base.items():
    m.fit(X_tr, y_tr); preds[name] = m.predict(X_te)
P = pd.DataFrame(preds)
print("基模型预测的两两相关性(越低越多样):")
print(P.corr().round(2).to_string())
print("→ 不同算法的预测相关性不高 → 它们犯不同的错 → 适合投票集成\n")

# 反例: 三个几乎一样的模型(同算法不同种子)投票 → 几乎无提升 / near-identical models don't help
from sklearn.ensemble import VotingClassifier
same = [(f"rf{i}", RandomForestClassifier(n_estimators=100, random_state=i)) for i in range(3)]
single_rf = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=0), wine.data, wine.target, cv=5).mean()
vote_same = cross_val_score(VotingClassifier(same, voting="soft"), wine.data, wine.target, cv=5).mean()
print(f"单个 RF: {single_rf:.3f}  vs  3 个相似 RF 投票: {vote_same:.3f}  (几乎无提升!)")
print("→ 多样性是集成有效的前提; 雷同的模型放一起投票没意义")


<a id="5"></a>
## 5. 加权投票 + 小结 ⭐ / Weighted Voting & Summary

如果某些模型明显更强，可以给它们**更大的投票权重**（`weights=[...]`）。但要注意：**集成不一定比最好的单模型强**——如果一个模型远超其他、或基模型互相高度相关，集成反而可能被弱模型拖累。所以集成是"通常有帮助"的稳健手段，但不是免费午餐，要**用 CV 验证**。
If some models are clearly stronger, give them **larger voting weights** (`weights=[...]`). But beware: **an ensemble doesn't always beat the best single model** — if one model dominates, or the bases are highly correlated, the ensemble can be dragged down. So voting is a robust "usually helps" tool, but not a free lunch — **validate with CV**.


In [ ]:
# 加权软投票: 给更强的模型更大权重 / weighted soft voting
weighted = VotingClassifier(estimators, voting="soft", weights=[1, 1, 2])   # RF 权重更大
print(f"等权软投票:   {cross_val_score(VotingClassifier(estimators, voting='soft'), wine.data, wine.target, cv=5).mean():.3f}")
print(f"加权软投票(RF×2): {cross_val_score(weighted, wine.data, wine.target, cv=5).mean():.3f}")
print(f"最好的单模型:  {max(cross_val_score(m, wine.data, wine.target, cv=5).mean() for m in base.values()):.3f}")
print("\n权重可调(给强模型更大权); 但集成不保证胜过最好单模型 → 务必用 CV 验证")


```
异质集成: 组合不同算法的模型(线性/核/树); 不同模型犯不同错 → 错误抵消
两个必要条件: ①各自够好(强于瞎猜) ②足够多样(犯不同错, 预测相关性低)
硬投票: 按类别多数; 软投票: 按概率平均(用上置信度, 通常更好, 需概率校准 7.9)
多样性是关键: 雷同模型投票无意义(实测相关性高则几乎无提升)
加权投票给强模型更大权; 但集成不保证胜过最好单模型 → CV 验证
```

### 💡 面试速查 / Interview cheat-sheet
1. **集成有效的两条件**: 各自够好 + 足够多样(犯不同错)。
   Ensembles need: each decent + diverse (different errors).
2. **硬投票(类别多数) vs 软投票(概率平均)**; 软投票通常更好。
   Hard voting (class majority) vs soft voting (probability average); soft is usually better.
3. **软投票需概率校准**(7.9)才发挥好。
   Soft voting works best with calibrated probabilities (7.9).
4. **多样性是关键**: 雷同模型投票无意义。
   Diversity is key: voting near-identical models is pointless.
5. **集成不保证胜过最好单模型** → 用 CV 验证。
   An ensemble isn't guaranteed to beat the best single model → validate with CV.

### 下一节 / Next
**8.7 Stacking & Blending**——比投票更强的组合: 不用固定规则平均, 而是训一个"元模型"去**学习怎么组合**各基模型的预测(用 OOF 预测防泄漏)。
**8.7 Stacking & Blending** — a stronger combination than voting: instead of a fixed averaging rule, train a "meta-model" to **learn how to combine** the base models' predictions (using OOF predictions to prevent leakage).
